## Setup e connessione a MongoDB Atlas

In [6]:
import sys
sys.path.append("..")
import os
import numpy as np
import pandas as pd
import sqlite3
from dotenv import load_dotenv
from pymongo import MongoClient

from src.frames import extract_frames
from src.metrics import curve_from_frames

In [7]:
# carichiamo la stringa di connessione da .env e ci colleghiamo
load_dotenv("../.env", override=True)
mongo_uri = os.getenv("MONGO_URI")

client = MongoClient(mongo_uri)
db = client["color_in_motion"]
frames_collection = db["frames"]

print("Connessione riuscita")
print("Database esistenti:", client.list_database_names())

Connessione riuscita
Database esistenti: ['color_in_motion', 'mongo_ted', 'sample_mflix', 'admin', 'local']


## Staging: scriviamo ogni fotogramma in MongoDB

Per ogni film, per ogni secondo del trailer, salviamo un documento con le
metriche di colore E il colore RGB pieno (usato per i movie barcode).
Questo e il dato grezzo, non ancora aggregato: lo staging vero del progetto.

In [8]:
# ripuliamo la collezione prima di riscriverla, per evitare duplicati
frames_collection.delete_many({})
print("Collezione pulita. Documenti ora:", frames_collection.count_documents({}))

Collezione pulita. Documenti ora: 0


In [9]:
# lista dei trailer di genere
cartella = "../data/raw/trailers"
files_trailer = []
for f in os.listdir(cartella):
    if f.endswith(".mp4"):
        files_trailer.append(f)

print("Trailer di genere da processare:", len(files_trailer))

Trailer di genere da processare: 133


In [10]:
# scriviamo in mongo un documento per ogni secondo di ogni trailer di genere
# includiamo anche il colore RGB pieno (mean_colour), non solo le 3 metriche
fatti = 0
for nome_file in files_trailer:
    tconst = nome_file.replace(".mp4", "")
    percorso = cartella + "/" + nome_file

    frames = extract_frames(percorso, per_second=1)
    brightness, temperature, saturation, colours = curve_from_frames(frames)

    for secondo in range(len(brightness)):
        colore_rgb = colours[secondo]
        documento = {
            "tconst": tconst,
            "secondo": secondo,
            "brightness": brightness[secondo],
            "temperature": temperature[secondo],
            "saturation": saturation[secondo],
            "mean_colour": [float(colore_rgb[0]), float(colore_rgb[1]), float(colore_rgb[2])],
            "gruppo": "genere",
        }
        frames_collection.insert_one(documento)

    fatti = fatti + 1
    if fatti % 20 == 0:
        print("Fatti", fatti, "film...")

print("Finito! Documenti totali nella collezione:", frames_collection.count_documents({}))

Fatti 20 film...
Fatti 40 film...
Fatti 60 film...
Fatti 80 film...
Fatti 100 film...
Fatti 120 film...
Finito! Documenti totali nella collezione: 17642


In [11]:
# stessa cosa per i 9 film d'autore
cartella_autore = "../data/raw/auteur_trailers"
files_autore = []
for f in os.listdir(cartella_autore):
    if f.endswith(".mp4"):
        files_autore.append(f)

print("Film d'autore da processare:", len(files_autore))

fatti = 0
for nome_file in files_autore:
    tmdb_id = nome_file.replace(".mp4", "")
    percorso = cartella_autore + "/" + nome_file

    frames = extract_frames(percorso, per_second=1)
    brightness, temperature, saturation, colours = curve_from_frames(frames)

    for secondo in range(len(brightness)):
        colore_rgb = colours[secondo]
        documento = {
            "tconst": tmdb_id,
            "secondo": secondo,
            "brightness": brightness[secondo],
            "temperature": temperature[secondo],
            "saturation": saturation[secondo],
            "mean_colour": [float(colore_rgb[0]), float(colore_rgb[1]), float(colore_rgb[2])],
            "gruppo": "autore",
        }
        frames_collection.insert_one(documento)

    fatti = fatti + 1
    print("Fatto:", tmdb_id)

print("Finito! Documenti totali nella collezione:", frames_collection.count_documents({}))

Film d'autore da processare: 9
Fatto: 110160
Fatto: 1955
Fatto: 660120
Fatto: 38
Fatto: 376386
Fatto: 394117
Fatto: 340485
Fatto: 265177
Fatto: 24469
Finito! Documenti totali nella collezione: 18740


## Aggregazione: MongoDB calcola media e deviazione standard per film

In [12]:
pipeline = [
    {
        "$group": {
            "_id": "$tconst",
            "brightness_media": {"$avg": "$brightness"},
            "brightness_std": {"$stdDevPop": "$brightness"},
            "temperature_media": {"$avg": "$temperature"},
            "temperature_std": {"$stdDevPop": "$temperature"},
            "saturation_media": {"$avg": "$saturation"},
            "saturation_std": {"$stdDevPop": "$saturation"},
            "n_secondi": {"$sum": 1},
        }
    }
]

risultati_mongo = list(frames_collection.aggregate(pipeline))
print("Film aggregati:", len(risultati_mongo))
print(risultati_mongo[0])

Film aggregati: 142
{'_id': 'tt2250912', 'brightness_media': 18.273917508096435, 'brightness_std': 11.697682614712916, 'temperature_media': 0.16914994930273497, 'temperature_std': 8.812189290854478, 'saturation_media': 0.40478499900416937, 'saturation_std': 0.200363337623363, 'n_secondi': 157}


## Warehouse: scriviamo il risultato aggregato in SQLite

In [13]:
# uniamo le medie di Mongo con titolo/genere/anno (dal Source)
mongo_df = pd.DataFrame(risultati_mongo)
mongo_df = mongo_df.rename(columns={"_id": "tconst"})

info_film = pd.read_csv("../data/source/film_with_trailers.csv")
info_film = info_film[["tconst", "primaryTitle", "genere_principale", "startYear"]]

info_autore = pd.read_csv("../data/source/auteur_films.csv")
info_autore = info_autore.rename(columns={"titolo": "primaryTitle", "anno": "startYear", "tmdb_id": "tconst"})
info_autore["tconst"] = info_autore["tconst"].astype(str)
info_autore["genere_principale"] = "Auteur"
info_autore = info_autore[["tconst", "primaryTitle", "genere_principale", "startYear"]]

tutte_info = pd.concat([info_film, info_autore], ignore_index=True)
tabella_finale = mongo_df.merge(tutte_info, on="tconst", how="left")

print("Righe finali:", len(tabella_finale))
tabella_finale.head()

Righe finali: 142


,tconst,brightness_media,brightness_std,temperature_media,temperature_std,saturation_media,saturation_std,n_secondi,primaryTitle,genere_principale,startYear
0,tt2250912,18.273918,11.697683,0.169150,8.812189,0.404785,0.200363,157,Spider-Man: Homecoming,Action,2017
1,tt1179904,14.742943,17.237610,0.739990,5.440895,0.440016,0.330474,63,Paranormal Activity,Horror,2007
2,tt3289956,11.682806,11.424742,1.343976,5.651540,0.318441,0.210377,151,The Autopsy of Jane Doe,Horror,2016
3,tt1454468,20.774569,17.333293,-0.103176,5.725139,0.263795,0.204602,149,Gravity,Drama,2013
4,265177,17.806199,10.328730,3.380779,8.520720,0.213109,0.167941,157,Mommy,Auteur,2014


In [14]:
conn = sqlite3.connect("../data/warehouse/color_in_motion.db")
tabella_finale.to_sql("films", conn, if_exists="replace", index=False)

print("Tabella 'films' in SQLite aggiornata da MongoDB")
print("Righe:", len(tabella_finale))

check = pd.read_sql_query("SELECT COUNT(*) as n FROM films", conn)
print(check)

Tabella 'films' in SQLite aggiornata da MongoDB
Righe: 142
     n
0  142


## Validazione: le medie di MongoDB coincidono con quelle calcolate in Python?

In [15]:
# confrontiamo con il calcolo fatto in Python nel notebook 03 (prima del refactoring)
# se coincidono, la pipeline Mongo -> SQLite e' corretta
check_cols = pd.read_sql_query("SELECT * FROM films LIMIT 1", conn)
print("Colonne nel warehouse:", check_cols.columns.tolist())

Colonne nel warehouse: ['tconst', 'brightness_media', 'brightness_std', 'temperature_media', 'temperature_std', 'saturation_media', 'saturation_std', 'n_secondi', 'primaryTitle', 'genere_principale', 'startYear']


## Query SQL sul warehouse (esempio RDBMS)

In [16]:
query = """
SELECT genere_principale,
       ROUND(AVG(brightness_media), 1) AS luminosita_media,
       COUNT(*) AS n_film
FROM films
WHERE genere_principale != 'Auteur'
GROUP BY genere_principale
ORDER BY luminosita_media
"""

risultato = pd.read_sql_query(query, conn)
print(risultato)

  genere_principale  luminosita_media  n_film
0            Horror              12.3      18
1             Crime              18.9      19
2            Action              19.4      72
3             Drama              20.4      11
4            Comedy              23.6      13
